# Wactorz in a notebook

The IMU example run from Jupyter: the system runs on the notebook's own event loop through `wactorz.serve()`, readings are published from a cell, anomalies are read back from a cell, and the running agent is inspected in-process.

Before running:

- a broker on `localhost:1883` (`docker compose up -d mosquitto` from the repository root);
- `pip install wactorz` in the kernel's environment;
- the kernel started with this folder as its working directory, so `agent.py` and `model.py` are importable;
- a free port for the dashboard: Jupyter takes 8888, the dashboard's default, so the notebook uses 8890.

In [ ]:
import asyncio
import json
import sys
from pathlib import Path

import aiomqtt

import wactorz

# Make this folder importable even when Jupyter was started elsewhere.
HERE = Path.cwd() if (Path.cwd() / "agent.py").exists() else Path(__file__).parent
sys.path.insert(0, str(HERE))
print("python", sys.version.split()[0], "| wactorz", wactorz.__version__)

## 1. Train the model

Stands in for your own training; it writes `imu_model.pkl` beside the agent.

In [ ]:
from train import main as train

train()

## 2. The agent, as a function

Before anything connects, the function is callable and testable on its own. `spec.build()` makes the actor around it, which here only supplies `options`, `persist` and `recall`.

In [ ]:
from agent import detect

spec = wactorz.spec_of(detect)
me = spec.build(persistence_dir=str(HERE / "state"))
print("name:", spec.name, "| subscribes:", spec.subscribes, "| publishes:", spec.publishes)
print("normal ->", detect({"ax": 0.1, "ay": 0.0, "az": 1.0}, me))
print("jolt   ->", detect({"ax": 9.0, "ay": -7.5, "az": 1.0}, me))

## 3. Start the system on the notebook's loop

`serve()` returns when the system stops, so it runs as a task. `minimal=True` means the monitor, the dashboard and our agent, no model key needed. The host keeps its signals; the task is cancelled to stop.

Jupyter itself listens on port 8888, which is also the dashboard's default, so the dashboard is put on another port here with `monitor_port`.

In [ ]:
DASHBOARD_PORT = 8890  # 8888 is Jupyter's own port

system_task = asyncio.create_task(
    wactorz.serve(
        agents=[detect], minimal=True, monitor_port=DASHBOARD_PORT, state_dir=str(HERE / "state")
    )
)
await asyncio.sleep(4)  # let the broker connection, the dashboard and the agent come up
print("running:", not system_task.done(), f"| dashboard: http://localhost:{DASHBOARD_PORT}/")

## 4. Listen for anomalies

A background task collects what the agent publishes on `anomalies/imu`.

In [ ]:
anomalies: list[dict] = []


async def collect() -> None:
    async with aiomqtt.Client("localhost", 1883) as client:
        await client.subscribe("anomalies/imu")
        async for message in client.messages:
            anomalies.append(json.loads(message.payload.decode()))


collector = asyncio.create_task(collect())
await asyncio.sleep(1)

## 5. Publish readings

Normal motion around gravity, with a jolt every 25th reading.

In [ ]:
import numpy as np

rng = np.random.default_rng()


async def publish_readings(count: int = 100, period: float = 0.05) -> None:
    async with aiomqtt.Client("localhost", 1883) as client:
        for i in range(count):
            if i % 25 == 24:
                reading = {"ax": 9.0, "ay": -7.5, "az": 1.0}
            else:
                ax, ay, az = rng.normal((0.0, 0.0, 1.0), (0.3, 0.3, 0.2))
                reading = {
                    "ax": round(float(ax), 3),
                    "ay": round(float(ay), 3),
                    "az": round(float(az), 3),
                }
            await client.publish("sensors/imu/notebook", json.dumps(reading))
            await asyncio.sleep(period)


await publish_readings()
await asyncio.sleep(1)
print(f"{len(anomalies)} anomalies (expected 4)")
for item in anomalies:
    print(item)

## 6. Inspect the running agent

The system runs in this process, so the actor itself is reachable: its counters, and what it persisted.

In [ ]:
registry = wactorz.system().registry

actor = registry.find_by_name("imu-anomaly")
print("messages processed:", actor.metrics.messages_processed)
print("anomalies persisted:", actor.recall("anomalies_total"))
print("state:", actor.state.value, "| uptime:", round(actor.metrics.uptime, 1), "s")

## 7. Ask it

What chat does with `@imu-anomaly {...}`, from a cell: `wactorz.ask` sends the reading as a task and returns the function's result. A reading that scores as normal returns `{"result": None}`; an agent that is not running raises `LookupError`, one that fails raises `RuntimeError`, one that never answers `asyncio.TimeoutError`.

In [ ]:
print("jolt   ->", await wactorz.ask("imu-anomaly", {"ax": 9.0, "ay": -7.5, "az": 1.0}))
print("normal ->", await wactorz.ask("imu-anomaly", {"ax": 0.1, "ay": 0.0, "az": 1.0}))

## 8. Stop

Cancelling the task runs the same shutdown as Ctrl-C would: agents stop, state is written, the dashboard closes.

In [ ]:
collector.cancel()
system_task.cancel()
await asyncio.gather(collector, system_task, return_exceptions=True)
print("stopped:", system_task.done())